<a href="https://colab.research.google.com/github/Karuneshtiwari/ML-lab/blob/main/Lab03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import minkowski

from google.colab import drive
drive.mount('/content/drive')

DATA_FILE = "/content/drive/MyDrive/ML_lab/Lab Session Data.xlsx"


def load_dataset(file_path):
    """
    Load the marketing_campaign sheet from the Excel file.
    """
    return pd.read_excel(file_path, sheet_name="marketing_campaign")

dataset = load_dataset(DATA_FILE)

print("Dataset Loaded Successfully!")
print("Shape :", dataset.shape)

display(dataset.head())

Mounted at /content/drive
Dataset Loaded Successfully!
Shape : (2240, 29)


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-04-09 00:00:00,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-08-03 00:00:00,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,21-08-2013,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-10-02 00:00:00,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,19-01-2014,94,173,...,5,0,0,0,0,0,0,3,11,0


In [2]:
# A1 : Identify Feature Datatypes

def identify_feature_types(dataframe):

    feature_information = []

    for column in dataframe.columns:

        dtype = dataframe[column].dtype

        if dtype == "object":
            measurement = "Nominal"

        elif "date" in column.lower() or "dt_" in column.lower():
            measurement = "Interval"

        elif dataframe[column].nunique() <= 15:
            measurement = "Ordinal"

        else:
            measurement = "Ratio"

        feature_information.append({

            "Feature": column,
            "Python Datatype": str(dtype),
            "Measurement Type": measurement

        })

    return pd.DataFrame(feature_information)

In [6]:
# A2 : Label Encoding and One-Hot Encoding

def label_encode_column(series):
    """
    Perform Label Encoding on a single categorical column.
    """

    unique_values = list(series.dropna().unique())

    mapping = {}

    encoded_values = []

    for index, value in enumerate(unique_values):
        mapping[value] = index

    for value in series:

        if pd.isna(value):
            encoded_values.append(np.nan)
        else:
            encoded_values.append(mapping[value])

    return encoded_values, mapping


def perform_label_encoding(dataframe):
    """
    Apply Label Encoding to all categorical columns.
    """

    encoded_dataframe = dataframe.copy()

    mappings = {}

    categorical_columns = dataframe.select_dtypes(include="object").columns

    for column in categorical_columns:

        encoded_values, mapping = label_encode_column(dataframe[column])

        encoded_dataframe[column] = encoded_values

        mappings[column] = mapping

    return encoded_dataframe, mappings


def one_hot_encode_column(dataframe, column_name):
    """
    Perform One-Hot Encoding on a single categorical column.
    """

    dataframe = dataframe.copy()

    unique_values = dataframe[column_name].dropna().unique()

    new_columns = {}

    for value in unique_values:

        new_column = f"{column_name}_{value}"

        new_columns[new_column] = (
            dataframe[column_name] == value
        ).astype(int)

    new_columns = pd.DataFrame(new_columns)

    dataframe = pd.concat(
        [
            dataframe.drop(columns=[column_name]),
            new_columns
        ],
        axis=1
    )

    return dataframe


def perform_one_hot_encoding(dataframe):
    """
    Apply One-Hot Encoding to all categorical columns.
    """

    encoded_dataframe = dataframe.copy()

    categorical_columns = encoded_dataframe.select_dtypes(
        include="object"
    ).columns

    for column in categorical_columns:

        encoded_dataframe = one_hot_encode_column(
            encoded_dataframe,
            column
        )

    return encoded_dataframe

In [7]:
# MAIN PROGRAM

# A1
print("=" * 70)
print("A1 : IDENTIFY FEATURE DATATYPES")
print("=" * 70)

feature_information = identify_feature_types(dataset)

display(feature_information)


# A2
print("\n" + "=" * 70)
print("A2 : LABEL ENCODING AND ONE-HOT ENCODING")
print("=" * 70)

label_encoded_dataset, label_mappings = perform_label_encoding(dataset)

print("\nLabel Encoding Mapping\n")

for column, mapping in label_mappings.items():

    print(f"{column} : {mapping}")

print("\nLabel Encoded Dataset")

display(label_encoded_dataset.head())

one_hot_encoded_dataset = perform_one_hot_encoding(dataset)

print("\nOne-Hot Encoded Dataset")

display(one_hot_encoded_dataset.head())



A1 : IDENTIFY FEATURE DATATYPES


,Feature,Python Datatype,Measurement Type
0,ID,int64,Ratio
1,Year_Birth,int64,Ratio
2,Education,object,Nominal
3,Marital_Status,object,Nominal
4,Income,float64,Ratio
5,Kidhome,int64,Ordinal
6,Teenhome,int64,Ordinal
7,Dt_Customer,object,Nominal
8,Recency,int64,Ratio
9,MntWines,int64,Ratio



A2 : LABEL ENCODING AND ONE-HOT ENCODING

Label Encoding Mapping

Education : {'Graduation': 0, 'PhD': 1, 'Master': 2, 'Basic': 3, '2n Cycle': 4}
Marital_Status : {'Single': 0, 'Together': 1, 'Married': 2, 'Divorced': 3, 'Widow': 4, 'Alone': 5, 'Absurd': 6, 'YOLO': 7}
Dt_Customer : {datetime.datetime(2012, 4, 9, 0, 0): 0, datetime.datetime(2014, 8, 3, 0, 0): 1, '21-08-2013': 2, datetime.datetime(2014, 10, 2, 0, 0): 3, '19-01-2014': 4, datetime.datetime(2013, 9, 9, 0, 0): 5, '13-11-2012': 6, datetime.datetime(2013, 8, 5, 0, 0): 7, datetime.datetime(2013, 6, 6, 0, 0): 8, '13-03-2014': 9, '15-11-2013': 10, datetime.datetime(2012, 10, 10, 0, 0): 11, '24-11-2012': 12, '24-12-2012': 13, '31-08-2012': 14, '28-03-2013': 15, datetime.datetime(2012, 3, 11, 0, 0): 16, datetime.datetime(2012, 8, 8, 0, 0): 17, datetime.datetime(2013, 6, 1, 0, 0): 18, '23-12-2012': 19, datetime.datetime(2014, 11, 1, 0, 0): 20, '18-03-2013': 21, datetime.datetime(2013, 2, 1, 0, 0): 22, '27-05-2013': 23, '20-02-2013'

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,0,0,58138.0,0,0,0,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,0,0,46344.0,1,1,1,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,0,1,71613.0,0,0,2,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,0,1,26646.0,1,0,3,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,1,2,58293.0,1,0,4,94,173,...,5,0,0,0,0,0,0,3,11,0



One-Hot Encoded Dataset


,ID,Year_Birth,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,...,Dt_Customer_27-03-2014,Dt_Customer_15-12-2012,Dt_Customer_17-09-2012,Dt_Customer_2013-02-06 00:00:00,Dt_Customer_21-12-2012,Dt_Customer_2013-01-11 00:00:00,Dt_Customer_2013-10-08 00:00:00,Dt_Customer_2012-11-10 00:00:00,Dt_Customer_20-12-2012,Dt_Customer_2014-09-01 00:00:00
0,5524,1957,58138.0,0,0,58,635,88,546,172,...,0,0,0,0,0,0,0,0,0,0
1,2174,1954,46344.0,1,1,38,11,1,6,2,...,0,0,0,0,0,0,0,0,0,0
2,4141,1965,71613.0,0,0,26,426,49,127,111,...,0,0,0,0,0,0,0,0,0,0
3,6182,1984,26646.0,1,0,26,11,4,20,10,...,0,0,0,0,0,0,0,0,0,0
4,5324,1981,58293.0,1,0,94,173,43,118,46,...,0,0,0,0,0,0,0,0,0,0
